# 🔍 Data Validation Notebook

This notebook performs comprehensive data validation across all tables in the VLM Router database.

## Tables Checked
- **vlm_samples**: Input samples (prompts, ground truth, metadata)
- **vlm_images**: Image binary data
- **vlm_responses**: Model outputs and scores (per sample × model)
- **vlm_evaluations**: Glider & Semantic F1 scores (per sample × model)

## Issues Detected
1. **Missing Evaluations**: Responses without corresponding evaluations
2. **Missing Latency**: Responses with NULL latency_ms
3. **Missing Response Samples**: Samples without any responses
4. **Missing Images**: Samples referencing non-existent images
5. **Missing Glider Scores**: Evaluations without glider_score
6. **Failed Responses**: Responses with ok=False or error messages
7. **Data Split Coverage**: Missing model coverage by split

In [1]:
# === Setup ===
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import text
from collections import defaultdict
from IPython.display import display, HTML, Markdown
import warnings
import sys

warnings.filterwarnings('ignore')

# Add artemis_final directory to path for imports
# Notebook is at: artemis_final/ares/notebooks/
# We need to add: artemis_final/ to import ares.db.connection
NOTEBOOK_DIR = Path.cwd()
ARES_DIR = NOTEBOOK_DIR.parent  # artemis_final/ares/
ARTEMIS_DIR = ARES_DIR.parent   # artemis_final/

if str(ARTEMIS_DIR) not in sys.path:
    sys.path.insert(0, str(ARTEMIS_DIR))

print(f"📁 Notebook directory: {NOTEBOOK_DIR}")
print(f"📁 Added to path: {ARTEMIS_DIR}")

📁 Notebook directory: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/notebooks/ares
📁 Added to path: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final


In [2]:
# === Import Database Connection ===
from ares.db.connection import get_engine, test_connection

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

In [3]:
# === Test Connection ===
print("🔗 Testing database connection...")
test_connection()
engine = get_engine()

🔗 Testing database connection...
✓ Connected to database. Server time: 2025-12-07 19:08:48.961612+00:00


---
## 📊 Table Overview: Row Counts

In [4]:
# === Get Row Counts for All Tables ===
tables = ['vlm_samples', 'vlm_images', 'vlm_responses', 'vlm_evaluations']
row_counts = {}

with engine.connect() as conn:
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        row_counts[table] = result.fetchone()[0]

print("📊 Table Row Counts:")
print("=" * 50)
for table, count in row_counts.items():
    print(f"{table:25} {count:>15,} rows")
print("=" * 50)

📊 Table Row Counts:
vlm_samples                        55,890 rows
vlm_images                         45,674 rows
vlm_responses                     279,450 rows
vlm_evaluations                   269,464 rows


In [5]:
# === Get Distinct Model Names ===
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT DISTINCT model_name, COUNT(*) as count 
        FROM vlm_responses 
        GROUP BY model_name 
        ORDER BY count DESC
    """))
    models_df = pd.DataFrame(result.fetchall(), columns=['model_name', 'response_count'])

print("\n🤖 Models in vlm_responses:")
display(models_df)
MODEL_NAMES = models_df['model_name'].tolist()
NUM_MODELS = len(MODEL_NAMES)
print(f"\nTotal unique models: {NUM_MODELS}")


🤖 Models in vlm_responses:


,model_name,response_count
0,deepseek_ocr,55890
1,gemma_3_27b,55890
2,qwen2_5_vl_3b,55890
3,qwen2_5_vl_7b,55890
4,qwen3_vl_8b_thinking,55890



Total unique models: 5


In [6]:
# === Get Data Splits ===
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT data_split, COUNT(*) as count 
        FROM vlm_samples 
        WHERE data_split IS NOT NULL
        GROUP BY data_split 
        ORDER BY count DESC
    """))
    splits_df = pd.DataFrame(result.fetchall(), columns=['data_split', 'sample_count'])

print("\n📂 Data Splits in vlm_samples:")
display(splits_df)
DATA_SPLITS = splits_df['data_split'].tolist()


📂 Data Splits in vlm_samples:


,data_split,sample_count
0,train,39224
1,val,8380
2,test,8286


In [7]:
# === Get Source Configs ===
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT source_config, COUNT(*) as count 
        FROM vlm_samples 
        GROUP BY source_config 
        ORDER BY count DESC
    """))
    configs_df = pd.DataFrame(result.fetchall(), columns=['source_config', 'sample_count'])

print("\n📁 Source Configs in vlm_samples:")
display(configs_df)
SOURCE_CONFIGS = configs_df['source_config'].tolist()
print(f"\nTotal unique configs: {len(SOURCE_CONFIGS)}")


📁 Source Configs in vlm_samples:


,source_config,sample_count
0,finqa,2142
1,dvqa,2012
2,figureqa,2011
3,datikz,2010
4,docvqa,2009
5,chart2text,1310
6,aokvqa,1310
7,chartqa,1270
8,ai2d,1260
9,cocoqa,1260



Total unique configs: 48


---
## ❌ Issue 1: Missing Evaluations

Find responses that don't have corresponding evaluations.

In [8]:
# === Missing Evaluations ===
query_missing_evals = """
SELECT 
    r.sample_id,
    r.model_name,
    s.source_config,
    s.data_split
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
LEFT JOIN vlm_evaluations e 
    ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE e.evaluation_id IS NULL
ORDER BY s.source_config, r.model_name
"""

with engine.connect() as conn:
    missing_evals_df = pd.read_sql(text(query_missing_evals), conn)

print(f"❌ Missing Evaluations: {len(missing_evals_df):,} response(s) without evaluations")
print("=" * 60)

if len(missing_evals_df) > 0:
    # Summary by model
    missing_by_model = missing_evals_df.groupby('model_name').size().reset_index(name='missing_count')
    print("\n📊 Missing by Model:")
    display(missing_by_model)
    
    # Summary by config
    missing_by_config = missing_evals_df.groupby('source_config').size().reset_index(name='missing_count')
    print("\n📊 Missing by Source Config:")
    display(missing_by_config.head(20))
    
    # Summary by split
    missing_by_split = missing_evals_df.groupby('data_split').size().reset_index(name='missing_count')
    print("\n📊 Missing by Data Split:")
    display(missing_by_split)
    
    # Sample of missing
    print("\n📝 Sample of Missing Evaluations (first 10):")
    display(missing_evals_df.head(10))
else:
    print("✅ All responses have corresponding evaluations!")

❌ Missing Evaluations: 9,986 response(s) without evaluations

📊 Missing by Model:


,model_name,missing_count
0,deepseek_ocr,9327
1,gemma_3_27b,169
2,qwen2_5_vl_3b,161
3,qwen2_5_vl_7b,162
4,qwen3_vl_8b_thinking,167



📊 Missing by Source Config:


,source_config,missing_count
0,ai2d,260
1,aokvqa,310
2,chart2text,310
3,chartqa,270
4,clevr,259
5,cocoqa,260
6,datikz,1010
7,diagram_image_to_text,125
8,docvqa,1009
9,dvqa,1012



📊 Missing by Data Split:


,data_split,missing_count
0,test,1477
1,train,7030
2,val,1479



📝 Sample of Missing Evaluations (first 10):


,sample_id,model_name,source_config,data_split
0,ai2d_770_0221db33,deepseek_ocr,ai2d,train
1,ai2d_771_63dca3d4,deepseek_ocr,ai2d,train
2,ai2d_772_98898cbb,deepseek_ocr,ai2d,val
3,ai2d_773_05cdced9,deepseek_ocr,ai2d,val
4,ai2d_788_981329a0,deepseek_ocr,ai2d,val
5,ai2d_789_c6ee2a57,deepseek_ocr,ai2d,test
6,ai2d_790_1707c5b5,deepseek_ocr,ai2d,train
7,ai2d_791_2190bf9c,deepseek_ocr,ai2d,val
8,ai2d_792_85e41904,deepseek_ocr,ai2d,test
9,ai2d_793_c3330854,deepseek_ocr,ai2d,train


---
## ⏱️ Issue 2: Missing Latency

Find responses with NULL or zero latency_ms.

In [9]:
# === Missing Latency ===
query_missing_latency = """
SELECT 
    r.sample_id,
    r.model_name,
    r.latency_ms,
    s.source_config,
    s.data_split,
    r.ok,
    r.error_message
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
WHERE r.latency_ms IS NULL OR r.latency_ms <= 0
ORDER BY s.source_config, r.model_name
"""

with engine.connect() as conn:
    missing_latency_df = pd.read_sql(text(query_missing_latency), conn)

print(f"⏱️ Missing Latency: {len(missing_latency_df):,} response(s) with NULL or zero latency")
print("=" * 60)

if len(missing_latency_df) > 0:
    # Summary by model
    latency_by_model = missing_latency_df.groupby('model_name').size().reset_index(name='missing_count')
    print("\n📊 Missing Latency by Model:")
    display(latency_by_model)
    
    # Summary by config
    latency_by_config = missing_latency_df.groupby('source_config').size().reset_index(name='missing_count')
    print("\n📊 Missing Latency by Source Config:")
    display(latency_by_config.head(20))
    
    # Check if these are failed responses
    failed_count = missing_latency_df[missing_latency_df['ok'] == False].shape[0]
    print(f"\n⚠️ Of these, {failed_count} have ok=False (failed responses)")
    
    # Sample
    print("\n📝 Sample of Missing Latency (first 10):")
    display(missing_latency_df.head(10))
else:
    print("✅ All responses have valid latency values!")

⏱️ Missing Latency: 9,986 response(s) with NULL or zero latency

📊 Missing Latency by Model:


,model_name,missing_count
0,deepseek_ocr,9327
1,gemma_3_27b,169
2,qwen2_5_vl_3b,161
3,qwen2_5_vl_7b,162
4,qwen3_vl_8b_thinking,167



📊 Missing Latency by Source Config:


,source_config,missing_count
0,ai2d,260
1,aokvqa,310
2,chart2text,310
3,chartqa,270
4,clevr,259
5,cocoqa,260
6,datikz,1010
7,diagram_image_to_text,125
8,docvqa,1009
9,dvqa,1012



⚠️ Of these, 9986 have ok=False (failed responses)

📝 Sample of Missing Latency (first 10):


,sample_id,model_name,latency_ms,source_config,data_split,ok,error_message
0,ai2d_953_5223bfc7,deepseek_ocr,0.0,ai2d,train,False,None
1,ai2d_901_09c26d6b,deepseek_ocr,0.0,ai2d,test,False,None
2,ai2d_885_ce10ba36,deepseek_ocr,0.0,ai2d,train,False,None
3,ai2d_889_d4407897,deepseek_ocr,0.0,ai2d,train,False,None
4,ai2d_888_60439175,deepseek_ocr,0.0,ai2d,val,False,None
5,ai2d_887_b67ffe67,deepseek_ocr,0.0,ai2d,train,False,None
6,ai2d_1_85c967d3,deepseek_ocr,0.0,ai2d,test,False,None
7,ai2d_913_05b94130,deepseek_ocr,0.0,ai2d,train,False,None
8,ai2d_952_c7c8513f,deepseek_ocr,0.0,ai2d,train,False,None
9,ai2d_950_6a706789,deepseek_ocr,0.0,ai2d,val,False,None


---
## 📝 Issue 3: Missing Response Samples

Find samples that don't have any responses recorded.

In [10]:
# === Missing Response Samples ===
query_missing_responses = """
SELECT 
    s.sample_id,
    s.source_config,
    s.data_split,
    s.router_task,
    s.created_at
FROM vlm_samples s
LEFT JOIN vlm_responses r ON s.sample_id = r.sample_id
WHERE r.response_id IS NULL
ORDER BY s.source_config
"""

with engine.connect() as conn:
    missing_responses_df = pd.read_sql(text(query_missing_responses), conn)

print(f"📝 Missing Responses: {len(missing_responses_df):,} sample(s) without any responses")
print("=" * 60)

if len(missing_responses_df) > 0:
    # Summary by config
    resp_by_config = missing_responses_df.groupby('source_config').size().reset_index(name='missing_count')
    print("\n📊 Missing Responses by Source Config:")
    display(resp_by_config.head(20))
    
    # Summary by split
    resp_by_split = missing_responses_df.groupby('data_split').size().reset_index(name='missing_count')
    print("\n📊 Missing Responses by Data Split:")
    display(resp_by_split)
    
    # Sample
    print("\n📝 Sample of Missing Responses (first 10):")
    display(missing_responses_df.head(10))
else:
    print("✅ All samples have at least one response!")

📝 Missing Responses: 0 sample(s) without any responses
✅ All samples have at least one response!


---
## 🖼️ Issue 4: Missing Images

Find samples that reference an image_id but the image doesn't exist in vlm_images.

In [11]:
# === Missing Images ===
query_missing_images = """
SELECT 
    s.sample_id,
    s.image_id,
    s.source_config,
    s.data_split,
    s.router_task
FROM vlm_samples s
LEFT JOIN vlm_images i ON s.image_id = i.image_id
WHERE s.image_id IS NOT NULL AND i.image_id IS NULL
ORDER BY s.source_config
"""

with engine.connect() as conn:
    missing_images_df = pd.read_sql(text(query_missing_images), conn)

print(f"🖼️ Missing Images: {len(missing_images_df):,} sample(s) referencing non-existent images")
print("=" * 60)

if len(missing_images_df) > 0:
    # Summary by config
    img_by_config = missing_images_df.groupby('source_config').size().reset_index(name='missing_count')
    print("\n📊 Missing Images by Source Config:")
    display(img_by_config.head(20))
    
    # Sample
    print("\n📝 Sample of Missing Images (first 10):")
    display(missing_images_df.head(10))
else:
    print("✅ All sample image references are valid!")

🖼️ Missing Images: 0 sample(s) referencing non-existent images
✅ All sample image references are valid!


---
## ⭐ Issue 5: Missing Glider Scores

Find evaluations that don't have glider_score populated.

In [12]:
# === Missing Glider Scores ===
query_missing_glider = """
SELECT 
    e.sample_id,
    e.model_name,
    e.glider_score,
    s.source_config,
    s.data_split
FROM vlm_evaluations e
JOIN vlm_samples s ON e.sample_id = s.sample_id
WHERE e.glider_score IS NULL
ORDER BY s.source_config, e.model_name
"""

with engine.connect() as conn:
    missing_glider_df = pd.read_sql(text(query_missing_glider), conn)

print(f"⭐ Missing Glider Scores: {len(missing_glider_df):,} evaluation(s) without glider_score")
print("=" * 60)

if len(missing_glider_df) > 0:
    # Summary by model
    glider_by_model = missing_glider_df.groupby('model_name').size().reset_index(name='missing_count')
    print("\n📊 Missing Glider by Model:")
    display(glider_by_model)
    
    # Summary by config
    glider_by_config = missing_glider_df.groupby('source_config').size().reset_index(name='missing_count')
    print("\n📊 Missing Glider by Source Config:")
    display(glider_by_config.head(20))
    
    # Sample
    print("\n📝 Sample of Missing Glider Scores (first 10):")
    display(missing_glider_df.head(10))
else:
    print("✅ All evaluations have glider scores!")

⭐ Missing Glider Scores: 0 evaluation(s) without glider_score
✅ All evaluations have glider scores!


---
## 🚨 Issue 6: Failed Responses

Find responses with ok=False or error messages.

In [13]:
# === Failed Responses ===
query_failed = """
SELECT 
    r.sample_id,
    r.model_name,
    r.ok,
    r.error_message,
    r.stop_reason,
    s.source_config,
    s.data_split
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
WHERE r.ok = FALSE OR r.error_message IS NOT NULL
ORDER BY s.source_config, r.model_name
"""

with engine.connect() as conn:
    failed_df = pd.read_sql(text(query_failed), conn)

print(f"🚨 Failed Responses: {len(failed_df):,} response(s) with ok=False or error_message")
print("=" * 60)

if len(failed_df) > 0:
    # Summary by model
    failed_by_model = failed_df.groupby('model_name').size().reset_index(name='failed_count')
    print("\n📊 Failed by Model:")
    display(failed_by_model)
    
    # Summary by config
    failed_by_config = failed_df.groupby('source_config').size().reset_index(name='failed_count')
    print("\n📊 Failed by Source Config:")
    display(failed_by_config.head(20))
    
    # Error types
    if failed_df['error_message'].notna().any():
        print("\n🔍 Sample Error Messages:")
        error_samples = failed_df[failed_df['error_message'].notna()]['error_message'].value_counts().head(10)
        display(error_samples)
    
    # Sample
    print("\n📝 Sample of Failed Responses (first 10):")
    display(failed_df.head(10))
else:
    print("✅ No failed responses found!")

🚨 Failed Responses: 9,986 response(s) with ok=False or error_message

📊 Failed by Model:


,model_name,failed_count
0,deepseek_ocr,9327
1,gemma_3_27b,169
2,qwen2_5_vl_3b,161
3,qwen2_5_vl_7b,162
4,qwen3_vl_8b_thinking,167



📊 Failed by Source Config:


,source_config,failed_count
0,ai2d,260
1,aokvqa,310
2,chart2text,310
3,chartqa,270
4,clevr,259
5,cocoqa,260
6,datikz,1010
7,diagram_image_to_text,125
8,docvqa,1009
9,dvqa,1012



🔍 Sample Error Messages:


error_message
Failed after 3 retries: Connection error.    559
cannot write mode CMYK as PNG                 35
Name: count, dtype: int64


📝 Sample of Failed Responses (first 10):


,sample_id,model_name,ok,error_message,stop_reason,source_config,data_split
0,ai2d_875_4387ec3b,deepseek_ocr,False,None,None,ai2d,train
1,ai2d_872_1ea2dc85,deepseek_ocr,False,None,None,ai2d,train
2,ai2d_877_fc545a9e,deepseek_ocr,False,None,None,ai2d,train
3,ai2d_878_6edc77e0,deepseek_ocr,False,None,None,ai2d,train
4,ai2d_879_3b33a2b9,deepseek_ocr,False,None,None,ai2d,train
5,ai2d_880_de866c18,deepseek_ocr,False,None,None,ai2d,train
6,ai2d_881_f1dc906a,deepseek_ocr,False,None,None,ai2d,test
7,ai2d_876_07514677,deepseek_ocr,False,None,None,ai2d,train
8,ai2d_883_1ab7bf3c,deepseek_ocr,False,None,None,ai2d,val
9,ai2d_882_477d5741,deepseek_ocr,False,None,None,ai2d,train


---
## 📈 Issue 7: Model Coverage by Split

Check if all models have responses for all samples in each split.

In [14]:
# === Model Coverage Analysis ===
query_coverage = """
SELECT 
    s.data_split,
    s.source_config,
    COUNT(DISTINCT s.sample_id) as total_samples,
    COUNT(DISTINCT r.model_name) as models_with_responses
FROM vlm_samples s
LEFT JOIN vlm_responses r ON s.sample_id = r.sample_id
WHERE s.data_split IS NOT NULL
GROUP BY s.data_split, s.source_config
ORDER BY s.data_split, s.source_config
"""

with engine.connect() as conn:
    coverage_df = pd.read_sql(text(query_coverage), conn)

print("📈 Model Coverage Summary:")
print("=" * 60)
display(coverage_df)

📈 Model Coverage Summary:


,data_split,source_config,total_samples,models_with_responses
0,test,ai2d,185,5
1,test,aokvqa,189,5
2,test,chart2text,217,5
3,test,chartqa,174,5
4,test,clevr,212,5
...,...,...,...,...
139,val,visualmrc,147,5
140,val,vqarad,51,5
141,val,vqav2,163,5
142,val,vsr,158,5


In [15]:
# === Detailed Model × Config Coverage ===
query_detailed_coverage = """
SELECT 
    s.source_config,
    r.model_name,
    s.data_split,
    COUNT(*) as response_count
FROM vlm_samples s
JOIN vlm_responses r ON s.sample_id = r.sample_id
WHERE s.data_split IS NOT NULL
GROUP BY s.source_config, r.model_name, s.data_split
ORDER BY s.source_config, r.model_name, s.data_split
"""

with engine.connect() as conn:
    detailed_coverage_df = pd.read_sql(text(query_detailed_coverage), conn)

# Create pivot table
pivot_coverage = detailed_coverage_df.pivot_table(
    index=['source_config', 'data_split'],
    columns='model_name',
    values='response_count',
    fill_value=0
)

print("\n📊 Detailed Model Coverage (responses per config/split/model):")
print("=" * 80)
display(pivot_coverage)


📊 Detailed Model Coverage (responses per config/split/model):


model_name                deepseek_ocr  gemma_3_27b  qwen2_5_vl_3b  \
source_config data_split                                             
ai2d          test               185.0        185.0          185.0   
              train              894.0        894.0          894.0   
              val                181.0        181.0          181.0   
aokvqa        test               189.0        189.0          189.0   
              train              912.0        912.0          912.0   
...                                ...          ...            ...   
vsr           train              747.0        747.0          747.0   
              val                158.0        158.0          158.0   
websight      test               171.0        171.0          171.0   
              train              743.0        743.0          743.0   
              val                155.0        155.0          155.0   

model_name                qwen2_5_vl_7b  qwen3_vl_8b_thinking  
source_config data_split                                       
ai2d          test                185.0                 185.0  
              train               894.0                 894.0  
              val                 181.0                 181.0  
aokvqa        test                189.0                 189.0  
              train               912.0                 912.0  
...                                 ...                   ...  
vsr           train               747.0                 747.0  
              val                 158.0                 158.0  
websight      test                171.0                 171.0  
              train               743.0                 743.0  
              val                 155.0                 155.0  

[144 rows x 5 columns]

In [16]:
# === Find Missing Model Coverage ===
# Get expected samples per config/split
query_expected = """
SELECT 
    source_config,
    data_split,
    COUNT(*) as expected_samples
FROM vlm_samples
WHERE data_split IS NOT NULL
GROUP BY source_config, data_split
"""

with engine.connect() as conn:
    expected_df = pd.read_sql(text(query_expected), conn)

# Compare with actual coverage
issues = []

for _, row in expected_df.iterrows():
    config = row['source_config']
    split = row['data_split']
    expected = row['expected_samples']
    
    for model in MODEL_NAMES:
        try:
            actual = pivot_coverage.loc[(config, split), model] if (config, split) in pivot_coverage.index else 0
        except:
            actual = 0
        
        if actual < expected:
            issues.append({
                'source_config': config,
                'data_split': split,
                'model_name': model,
                'expected': expected,
                'actual': actual,
                'missing': expected - actual
            })

missing_coverage_df = pd.DataFrame(issues)

if len(missing_coverage_df) > 0:
    print(f"\n⚠️ Missing Model Coverage: {len(missing_coverage_df):,} gaps found")
    print("=" * 60)
    
    # Summary by model
    missing_by_model = missing_coverage_df.groupby('model_name')['missing'].sum().reset_index()
    missing_by_model.columns = ['model_name', 'total_missing']
    print("\n📊 Total Missing by Model:")
    display(missing_by_model)
    
    # Top gaps
    print("\n📝 Top 20 Coverage Gaps:")
    display(missing_coverage_df.sort_values('missing', ascending=False).head(20))
else:
    print("\n✅ Complete model coverage for all config/split combinations!")


✅ Complete model coverage for all config/split combinations!


---
## 📋 Issue 8: Evaluation Coverage by Split

Check if all responses have evaluations for each split.

In [17]:
# === Evaluation Coverage by Config and Split ===
query_eval_coverage = """
SELECT 
    s.source_config,
    s.data_split,
    r.model_name,
    COUNT(r.response_id) as total_responses,
    COUNT(e.evaluation_id) as total_evaluations,
    COUNT(r.response_id) - COUNT(e.evaluation_id) as missing_evaluations
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE s.data_split IS NOT NULL
GROUP BY s.source_config, s.data_split, r.model_name
HAVING COUNT(r.response_id) > COUNT(e.evaluation_id)
ORDER BY (COUNT(r.response_id) - COUNT(e.evaluation_id)) DESC
"""

with engine.connect() as conn:
    eval_coverage_df = pd.read_sql(text(query_eval_coverage), conn)

if len(eval_coverage_df) > 0:
    print(f"📋 Evaluation Coverage Gaps: {len(eval_coverage_df):,} config/split/model combinations with missing evals")
    print("=" * 80)
    
    # Total missing
    total_missing = eval_coverage_df['missing_evaluations'].sum()
    print(f"\n🔢 Total missing evaluations: {total_missing:,}")
    
    # Summary by model
    missing_by_model = eval_coverage_df.groupby('model_name')['missing_evaluations'].sum().reset_index()
    print("\n📊 Missing Evaluations by Model:")
    display(missing_by_model)
    
    # Summary by split
    missing_by_split = eval_coverage_df.groupby('data_split')['missing_evaluations'].sum().reset_index()
    print("\n📊 Missing Evaluations by Split:")
    display(missing_by_split)
    
    # Top gaps
    print("\n📝 Top 20 Evaluation Coverage Gaps:")
    display(eval_coverage_df.head(20))
else:
    print("✅ All responses have corresponding evaluations!")

📋 Evaluation Coverage Gaps: 196 config/split/model combinations with missing evals

🔢 Total missing evaluations: 9,986

📊 Missing Evaluations by Model:


,model_name,missing_evaluations
0,deepseek_ocr,9327
1,gemma_3_27b,169
2,qwen2_5_vl_3b,161
3,qwen2_5_vl_7b,162
4,qwen3_vl_8b_thinking,167



📊 Missing Evaluations by Split:


,data_split,missing_evaluations
0,test,1477
1,train,7030
2,val,1479



📝 Top 20 Evaluation Coverage Gaps:


,source_config,data_split,model_name,total_responses,total_evaluations,missing_evaluations
0,finqa,train,deepseek_ocr,1516,695,821
1,datikz,train,deepseek_ocr,1434,698,736
2,docvqa,train,deepseek_ocr,1403,688,715
3,dvqa,train,deepseek_ocr,1381,670,711
4,figureqa,train,deepseek_ocr,1382,697,685
5,chart2text,train,deepseek_ocr,906,688,218
6,aokvqa,train,deepseek_ocr,912,697,215
7,chartqa,train,deepseek_ocr,887,690,197
8,ai2d,train,deepseek_ocr,894,705,189
9,cocoqa,train,deepseek_ocr,892,710,182


---
## 📊 Summary Report

In [18]:
# === Generate Summary Report ===
print("\n" + "=" * 80)
print("📊 DATA VALIDATION SUMMARY REPORT")
print("=" * 80)

report = {
    'Table Counts': {
        'vlm_samples': row_counts.get('vlm_samples', 0),
        'vlm_images': row_counts.get('vlm_images', 0),
        'vlm_responses': row_counts.get('vlm_responses', 0),
        'vlm_evaluations': row_counts.get('vlm_evaluations', 0),
    },
    'Issues Found': {
        '❌ Missing Evaluations': len(missing_evals_df),
        '⏱️ Missing Latency': len(missing_latency_df),
        '📝 Missing Response Samples': len(missing_responses_df),
        '🖼️ Missing Images': len(missing_images_df),
        '⭐ Missing Glider Scores': len(missing_glider_df),
        '🚨 Failed Responses': len(failed_df),
    }
}

print("\n📊 Table Row Counts:")
for table, count in report['Table Counts'].items():
    print(f"   {table:25} {count:>12,} rows")

print("\n🔍 Issues Found:")
total_issues = 0
for issue, count in report['Issues Found'].items():
    status = "✅" if count == 0 else "⚠️"
    print(f"   {status} {issue:35} {count:>10,}")
    total_issues += count

print(f"\n{'='*80}")
if total_issues == 0:
    print("🎉 NO ISSUES FOUND! All data is valid.")
else:
    print(f"⚠️ TOTAL ISSUES FOUND: {total_issues:,}")
print("=" * 80)


📊 DATA VALIDATION SUMMARY REPORT

📊 Table Row Counts:
   vlm_samples                     55,890 rows
   vlm_images                      45,674 rows
   vlm_responses                  279,450 rows
   vlm_evaluations                269,464 rows

🔍 Issues Found:
   ⚠️ ❌ Missing Evaluations                    9,986
   ⚠️ ⏱️ Missing Latency                       9,986
   ✅ 📝 Missing Response Samples                   0
   ✅ 🖼️ Missing Images                            0
   ✅ ⭐ Missing Glider Scores                      0
   ⚠️ 🚨 Failed Responses                       9,986

⚠️ TOTAL ISSUES FOUND: 29,958


In [19]:
# === Export Issues to CSV (optional) ===
export_issues = False  # Set to True to export

if export_issues:
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if len(missing_evals_df) > 0:
        missing_evals_df.to_csv(f'issues_missing_evals_{timestamp}.csv', index=False)
        print(f"Exported missing evaluations to issues_missing_evals_{timestamp}.csv")
    
    if len(missing_latency_df) > 0:
        missing_latency_df.to_csv(f'issues_missing_latency_{timestamp}.csv', index=False)
        print(f"Exported missing latency to issues_missing_latency_{timestamp}.csv")
    
    if len(missing_responses_df) > 0:
        missing_responses_df.to_csv(f'issues_missing_responses_{timestamp}.csv', index=False)
        print(f"Exported missing responses to issues_missing_responses_{timestamp}.csv")
    
    if len(failed_df) > 0:
        failed_df.to_csv(f'issues_failed_responses_{timestamp}.csv', index=False)
        print(f"Exported failed responses to issues_failed_responses_{timestamp}.csv")
else:
    print("ℹ️ Set export_issues = True to export issues to CSV files")

ℹ️ Set export_issues = True to export issues to CSV files


---
## 🔧 Quick Fix Queries

Below are SQL queries you can run to fix common issues.

In [20]:
# === Quick Fix Queries (Reference Only) ===

print("""
🔧 QUICK FIX REFERENCE QUERIES
================================

1. Get sample_ids for missing evaluations (to re-run evaluation):
--------------------------------------------------------------
SELECT r.sample_id, r.model_name
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE e.evaluation_id IS NULL;

2. Get samples with missing responses (to re-run inference):
------------------------------------------------------------
SELECT s.sample_id, s.source_config
FROM vlm_samples s
LEFT JOIN vlm_responses r ON s.sample_id = r.sample_id
WHERE r.response_id IS NULL;

3. Delete failed responses (to allow retry):
--------------------------------------------
DELETE FROM vlm_responses WHERE ok = FALSE;

4. Check orphaned evaluations (evaluations without responses):
--------------------------------------------------------------
SELECT e.*
FROM vlm_evaluations e
LEFT JOIN vlm_responses r ON e.sample_id = r.sample_id AND e.model_name = r.model_name
WHERE r.response_id IS NULL;

5. Reset evaluation progress for a specific config:
---------------------------------------------------
-- Delete evaluations for config, then re-run evaluation pipeline
DELETE FROM vlm_evaluations 
WHERE sample_id IN (SELECT sample_id FROM vlm_samples WHERE source_config = 'CONFIG_NAME');
""")


🔧 QUICK FIX REFERENCE QUERIES

1. Get sample_ids for missing evaluations (to re-run evaluation):
--------------------------------------------------------------
SELECT r.sample_id, r.model_name
FROM vlm_responses r
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE e.evaluation_id IS NULL;

2. Get samples with missing responses (to re-run inference):
------------------------------------------------------------
SELECT s.sample_id, s.source_config
FROM vlm_samples s
LEFT JOIN vlm_responses r ON s.sample_id = r.sample_id
WHERE r.response_id IS NULL;

3. Delete failed responses (to allow retry):
--------------------------------------------
DELETE FROM vlm_responses WHERE ok = FALSE;

4. Check orphaned evaluations (evaluations without responses):
--------------------------------------------------------------
SELECT e.*
FROM vlm_evaluations e
LEFT JOIN vlm_responses r ON e.sample_id = r.sample_id AND e.model_name = r.model_name
WHERE r.response_id 